In [1]:
import cv2
import toml
import msgpack_numpy as mpn
import msgpack as mp
import numpy as np
import os
from cv2 import aruco
from pd_multi_support import *
import polars as pl

Path definitions

In [2]:
_pth = os.path.dirname(os.getcwd())
_parent_folder = "data"
_fov = "120_fov"
permute_value = 300

useFisheye = True

_calib_folder_name = "calib_mono_120fov_1280_800_2026"
# _recording_folder_name = "3marker_linear_2d_160fov_t1"
_recording_folder_name = "linear_t1_may_14"

_webcam_calib_folder = os.path.join(
    _pth,'..', _parent_folder, "calibration",_fov, _calib_folder_name
)

_reference_recording_folder = os.path.join(
    _pth, '..', _parent_folder, "recordings",_fov,'noark_plus_data_may_14', _recording_folder_name
)
_reference_file = os.path.join(_reference_recording_folder, "webcam_color.msgpack")

_timestamp_file = os.path.join(_reference_recording_folder, "webcam_timestamp.msgpack")
with open(_timestamp_file, "rb") as f:
    _metadata = list(mp.Unpacker(f, object_hook=mpn.decode))
    _timestamp = np.array(_metadata)[:,1]
    _sync_pulse = np.array(_metadata)[:,0]

In [10]:
calib_data = toml.load('good.toml')

In [28]:
# calib_data

In [11]:
_timestamp

array(['2026-05-13 19:08:22.928182', '2026-05-13 19:08:22.947313',
       '2026-05-13 19:08:22.957312', ..., '2026-05-13 19:09:09.654388',
       '2026-05-13 19:09:09.662768', '2026-05-13 19:09:09.673021'],
      shape=(4672,), dtype='<U26')

In [12]:
_video_pth = _reference_file
_video_file = open(_video_pth, "rb")
_video_data = mp.Unpacker(_video_file, object_hook=mpn.decode)
_video_length = 0

for _frame in _video_data:
    _video_length += 1

_video_file.close()

print('video length, ', _video_length)

video length,  4672


In [13]:
_frame.shape

(800, 1280)

In [14]:
ARUCO_PARAMETERS = aruco.DetectorParameters()
ARUCO_DICT = aruco.getPredefinedDictionary(aruco.DICT_APRILTAG_36h11)
detector = aruco.ArucoDetector(ARUCO_DICT, ARUCO_PARAMETERS)
markerLength = 0.048
markerSeperation = 0.01

board = aruco.GridBoard(
    size=[1, 1],
    markerLength=markerLength,
    markerSeparation=markerSeperation,
    dictionary=ARUCO_DICT,
)

def estimate_pose_single_markers(
    corners, marker_size, camera_matrix, distortion_coefficients = np.zeros((5, 1))
):
    marker_points = np.array(
        [
            [-marker_size / 2, marker_size / 2, 0],
            [marker_size / 2, marker_size / 2, 0],
            [marker_size / 2, -marker_size / 2, 0],
            [-marker_size / 2, -marker_size / 2, 0],
        ],
        dtype=np.float32,
    )
    rvecs, tvecs = [], []
    for corner in corners:
        _, r, t = cv2.solvePnP(
            marker_points,
            corner,
            camera_matrix,
            distortion_coefficients,
            flags=cv2.SOLVEPNP_ITERATIVE,
        )
        if r is not None and t is not None:
            rvecs.append(r.reshape(1, 3).tolist())
            tvecs.append(t.reshape(1, 3).tolist())
        else:
            rvecs.append(np.array([[np.nan, np.nan, np.nan]]).tolist())
            tvecs.append(np.array([[np.nan, np.nan, np.nan]]).tolist())
    return np.array(rvecs, dtype=np.float32), np.array(tvecs, dtype=np.float32)

In [22]:
_video_file = open(_video_pth, "rb")
_video_data = mp.Unpacker(_video_file, object_hook=mpn.decode)

new_camera_matrix = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
    np.array(calib_data["calibration"]["camera_matrix"], dtype=np.float64),
    np.array(calib_data["calibration"]["dist_coeffs"], dtype=np.float64),
    (1280, 800),
    np.eye(3, dtype=np.float64),
    np.eye(3, dtype=np.float64),
    balance=1.0,
)

K = np.array(calib_data["calibration"]["camera_matrix"], dtype=np.float64)
D = np.array(calib_data["calibration"]["dist_coeffs"], dtype=np.float64)

map1, map2 = cv2.fisheye.initUndistortRectifyMap(
    K,
    D,
    np.eye(3, dtype=np.float64),
    new_camera_matrix,
    (1280, 800),
    cv2.CV_16SC2,
)

ar_dict = {'12_tvec':[], '12_rvec':[]}

for _frame in _video_data:
    _undistorted_frame = cv2.remap(_frame, map1, map2, interpolation=cv2.INTER_LINEAR) 


    _corners, _ids, _rejected = detector.detectMarkers(_undistorted_frame)
    _corners, _ids, _rejected, _ = detector.refineDetectedMarkers(
        _undistorted_frame, board, 
        _corners, _ids, _rejected
    )
    if _ids is not None:
        _rvecs, _tvecs = estimate_pose_single_markers(
            _corners, markerLength, new_camera_matrix, np.zeros((5, 1))
        )
        if 12 in _ids:
            idx = np.where(_ids == 12)[0][0]
            ar_dict['12_tvec'].append(_tvecs[idx].tolist())
            ar_dict['12_rvec'].append(_rvecs[idx].tolist())
        else:
            ar_dict['12_tvec'].append(np.array([[np.nan, np.nan, np.nan]]).tolist())
            ar_dict['12_rvec'].append(np.array([[np.nan, np.nan, np.nan]]).tolist())

_video_file.close()



In [27]:
ar_12_tvec = np.array(ar_dict['12_tvec']).reshape(-1, 3)
ar_12_rvec = np.array(ar_dict['12_rvec']).reshape(-1, 3)

In [6]:
ar_df = {"time": _timestamp, "sync": _sync_pulse}
ar_df = pl.from_dict(ar_df)
if type(ar_df["time"][0]) is not datetime:
    ar_df = ar_df.with_columns(pl.col("time").str.to_datetime())

In [27]:
tr = get_rb_marker_name(2)
tl = get_rb_marker_name(6)
br = get_rb_marker_name(4)
bl = get_rb_marker_name(8)

# tr = get_rb_marker_name(8)
# tl = get_rb_marker_name(6)
# br = get_rb_marker_name(2)
# bl = get_rb_marker_name(5)

# tr = get_rb_marker_name(4)
# tl = get_rb_marker_name(2)
# br = get_rb_marker_name(3)
# bl = get_rb_marker_name(1)

In [8]:
ar_df = ar_df.with_columns(
    pl.col("sync").cast(pl.Int8).cast(pl.Boolean)
)
ar_df['sync'][0]

False

In [7]:
start_pulse = ar_df["sync"].arg_true().head(1).item()
offset = (~ar_df["sync"].slice(start_pulse)).arg_true().head(1).item()

end_pulse = start_pulse + offset

print(f"Start pulse: {start_pulse}, End pulse: {end_pulse}")
ar_df = ar_df[start_pulse:end_pulse]
ar_corners = ar_results['corners'][start_pulse:end_pulse]
ids = ar_results['ids'][start_pulse:end_pulse]

_time_diff = mocap_df["time"][0] - ar_df["time"][0]

ar_df = ar_df.with_columns([(pl.col("time") + _time_diff).alias("time")])

SchemaError: invalid series dtype: expected `Boolean`, got `str` for series with name `sync`

This error occurred in the following expression:
	col("sync").arg_where()


In [ ]:
mocap_mean = {"x": [], "y": [], "z": []}
mocap_mean["x"] = mocap_df[[tr["x"], tl["x"], br["x"], bl["x"]]].to_numpy().mean(axis=1)
mocap_mean["y"] = mocap_df[[tr["y"], tl["y"], br["y"], bl["y"]]].to_numpy().mean(axis=1)
mocap_mean["z"] = mocap_df[[tr["z"], tl["z"], br["z"], bl["z"]]].to_numpy().mean(axis=1)

mocap_qt_0 = mocap_df[["rb_ang_x", "rb_ang_y", "rb_ang_z", "rb_ang_w"]][0].to_numpy()

mocap_rotation = R.from_quat(mocap_qt_0).as_matrix()

mocap_mean = pl.from_dict(mocap_mean)

mt_dict = {"x": [], "y": [], "z": []}
rmat_m = mocap_rotation[0]

for i in range(len(mocap_df["time"])):
    tvec_ar = rmat_m.T @ (
        mocap_mean[["x", "y", "z"]][i].to_numpy().reshape(3, 1)
        - mocap_mean[["x", "y", "z"]][0].to_numpy().reshape(3, 1)
    )
    tvec_ar = tvec_ar.T[0]
    mt_dict["x"].append(tvec_ar[0])
    mt_dict["y"].append(tvec_ar[1])
    mt_dict["z"].append(tvec_ar[2])

mt_dict["time"] = mocap_df["time"]

In [ ]:
mc_angle_arr = mocap_df[["rb_ang_x", "rb_ang_y", "rb_ang_z", "rb_ang_w"]].to_numpy()
mocap_angle = []
mc_ang_x = []
mc_ang_y = []
mc_ang_z = []
for _a in mc_angle_arr:
    try:
        _ax, _ay, _az = R.from_matrix(
            mocap_rotation[0].T @ R.from_quat(_a).as_matrix()
        ).as_euler("xyz", degrees=True)
        mc_ang_x.append(_ax)
        mc_ang_y.append(_ay)
        mc_ang_z.append(_az)
    except:
        _ax, _ay, _az = R.from_matrix(mocap_rotation[0].T @ np.eye(3)).as_euler(
            "xyz", degrees=True
        )
        mc_ang_x.append(_ax)
        mc_ang_y.append(_ay)
        mc_ang_z.append(_az)

In [ ]:
mocap = pl.from_dict(mt_dict)

x1 = interp1d(mocap["time"].dt.epoch(), mocap["x"], fill_value="extrapolate")
y1 = interp1d(mocap["time"].dt.epoch(), mocap["y"], fill_value="extrapolate")
z1 = interp1d(mocap["time"].dt.epoch(), mocap["z"], fill_value="extrapolate")

ax = interp1d(mocap["time"].dt.epoch(), mc_ang_x, fill_value="extrapolate")
ay = interp1d(mocap["time"].dt.epoch(), mc_ang_y, fill_value="extrapolate")
az = interp1d(mocap["time"].dt.epoch(), mc_ang_z, fill_value="extrapolate")

mocap_ip = {"time": ar_df["time"]}
mocap_ip["x"] = x1(ar_df["time"].dt.epoch())
mocap_ip["y"] = y1(ar_df["time"].dt.epoch())
mocap_ip["z"] = z1(ar_df["time"].dt.epoch())
mocap_ip["rx"] = ax(ar_df["time"].dt.epoch())
mocap_ip["ry"] = ay(ar_df["time"].dt.epoch())
mocap_ip["rz"] = az(ar_df["time"].dt.epoch())

mocap_ip = pl.from_dict(mocap_ip)

In [ ]:
default_ids = [12, 14, 20]